# WebNLG IE evaluation metrics (swap-aware)

This notebook contains the full evaluation code, including strict, soft, and swap-aware metrics.


In [1]:
import ast
import glob
import json
import re
import unicodedata
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd
from tqdm.auto import tqdm

pd.set_option("display.max_columns", 200)
pd.set_option("display.max_colwidth", 200)

# -----------------------
# Configuration
# -----------------------
CSV_GLOB = "./generations__*.csv"   # change if needed
OUTPUT_DIR = Path("./evaluation_outputs_ie")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

FILTER_SPLITS = None   # e.g. ["test"]
ACCENT_INSENSITIVE = True
LOWERCASE_MATCH = True
STRIP_ARTICLES = False
USE_SOFT_MATCH = True
SOFT_MATCH_THRESHOLD = 0.70
USE_SWAP_AWARE_MATCH = True


# -----------------------
# Parsing + normalization helpers
# -----------------------
ARTICLES = {
    "en": {"a", "an", "the"},
    "es": {"el", "la", "los", "las", "un", "una", "unos", "unas"},
    "ca": {"el", "la", "els", "les", "un", "una", "uns", "unes"},
    "gl": {"o", "a", "os", "as", "un", "unha", "uns", "unhas"},
    "eu": set(),
}


def normalize_text(text: str) -> str:
    if text is None:
        return ""
    text = str(text)
    text = text.replace("\n", " ").replace("\t", " ")
    return re.sub(r"\s+", " ", text).strip()


def strip_accents(text: str) -> str:
    text = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in text if not unicodedata.combining(ch))


def normalize_token_text(text: str, lang: str = "") -> str:
    txt = normalize_text(text)
    if LOWERCASE_MATCH:
        txt = txt.lower()
    if ACCENT_INSENSITIVE:
        txt = strip_accents(txt)
    txt = txt.replace("_", " ")
    txt = re.sub(r"\s+", " ", txt).strip()
    if STRIP_ARTICLES and lang in ARTICLES:
        toks = txt.split()
        while toks and toks[0] in ARTICLES[lang]:
            toks = toks[1:]
        txt = " ".join(toks)
    return txt


def parse_jsonish(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    if isinstance(value, (list, dict)):
        return value
    s = str(value).strip()
    if not s:
        return None
    try:
        return json.loads(s)
    except Exception:
        try:
            return ast.literal_eval(s)
        except Exception:
            return None


def parse_triple_text(triple_text: str):
    txt = normalize_text(triple_text).strip("[]")
    parts = [p.strip() for p in re.split(r"\s*\|\s*", txt)]
    if len(parts) < 3:
        return None
    return {
        "subject": parts[0],
        "predicate": parts[1],
        "object": " | ".join(parts[2:]),
    }


def parse_triples_cell(value):
    raw = parse_jsonish(value)
    triples = []
    if isinstance(raw, list):
        for item in raw:
            if isinstance(item, str):
                t = parse_triple_text(item)
                if t:
                    triples.append(t)
            elif isinstance(item, dict):
                if {"subject", "predicate", "object"}.issubset(item.keys()):
                    triples.append({
                        "subject": normalize_text(item["subject"]),
                        "predicate": normalize_text(item["predicate"]),
                        "object": normalize_text(item["object"]),
                    })
                elif {"s", "p", "o"}.issubset(item.keys()):
                    triples.append({
                        "subject": normalize_text(item["s"]),
                        "predicate": normalize_text(item["p"]),
                        "object": normalize_text(item["o"]),
                    })
    elif isinstance(raw, str):
        t = parse_triple_text(raw)
        if t:
            triples.append(t)
    return triples


def parse_bracketed_triples_text(value):
    txt = normalize_text(value)
    if not txt:
        return []
    matches = re.findall(r"\[(.*?)\]", txt, flags=re.DOTALL)
    triples = []
    for m in matches:
        t = parse_triple_text(m)
        if t:
            triples.append(t)
    return triples


def choose_predicted_triples(row):
    if "predicted_triples" in row and pd.notna(row["predicted_triples"]):
        triples = parse_triples_cell(row["predicted_triples"])
        if triples:
            return triples
    if "extracted_triples_text" in row and pd.notna(row["extracted_triples_text"]):
        triples = parse_bracketed_triples_text(row["extracted_triples_text"])
        if triples or normalize_text(row["extracted_triples_text"]):
            return triples
    if "raw_generation" in row and pd.notna(row["raw_generation"]):
        return parse_bracketed_triples_text(row["raw_generation"])
    return []


def triple_to_text(triple):
    return f"[{triple['subject']} | {triple['predicate']} | {triple['object']}]"


def normalize_triple(triple, lang=""):
    return (
        normalize_token_text(triple["subject"], lang),
        normalize_token_text(triple["predicate"], lang),
        normalize_token_text(triple["object"], lang),
    )


def normalize_triple_swapped(triple, lang=""):
    s, p, o = normalize_triple(triple, lang)
    return (o, p, s)


def load_generation_csvs(csv_glob=CSV_GLOB):
    paths = sorted(glob.glob(csv_glob))
    if not paths:
        raise FileNotFoundError(f"No CSV files found for pattern: {csv_glob}")

    frames = []
    for path in paths:
        frame = pd.read_csv(path)
        frame["source_csv"] = Path(path).name
        if "model_name" not in frame.columns:
            stem = Path(path).stem
            frame["model_name"] = stem.replace("generations__", "").replace("__", "/")
        frames.append(frame)

    data = pd.concat(frames, ignore_index=True)

    if FILTER_SPLITS is not None and "split" in data.columns:
        data = data[data["split"].isin(FILTER_SPLITS)].copy()

    data["gold_triples"] = data["triples"].apply(parse_triples_cell)
    data["pred_triples"] = data.apply(choose_predicted_triples, axis=1)
    data["gold_triples_text"] = data["gold_triples"].apply(lambda xs: [triple_to_text(x) for x in xs])
    data["pred_triples_text"] = data["pred_triples"].apply(lambda xs: [triple_to_text(x) for x in xs])
    data["num_gold_triples"] = data["gold_triples"].apply(len)
    data["num_pred_triples"] = data["pred_triples"].apply(len)
    return data


# -----------------------
# IE metrics
# -----------------------
def prf(tp, pred_total, gold_total):
    precision = tp / pred_total if pred_total else 0.0
    recall = tp / gold_total if gold_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    return float(precision), float(recall), float(f1)


def safe_div(num, den, default=0.0):
    return float(num / den) if den else float(default)


def exact_triple_overlap(pred_triples, gold_triples, lang=""):
    pred_norm = [normalize_triple(t, lang) for t in pred_triples]
    gold_norm = [normalize_triple(t, lang) for t in gold_triples]

    pred_counter = Counter(pred_norm)
    gold_counter = Counter(gold_norm)
    tp = sum((pred_counter & gold_counter).values())
    pred_total = sum(pred_counter.values())
    gold_total = sum(gold_counter.values())
    precision, recall, f1 = prf(tp, pred_total, gold_total)

    fp = pred_total - tp
    fn = gold_total - tp
    return {
        "triple_exact_tp": tp,
        "triple_exact_fp": fp,
        "triple_exact_fn": fn,
        "triple_exact_precision": precision,
        "triple_exact_recall": recall,
        "triple_exact_f1": f1,
        "pred_total": pred_total,
        "gold_total": gold_total,
    }


def exact_triple_overlap_swap_aware(pred_triples, gold_triples, lang=""):
    pred_norm = [normalize_triple(t, lang) for t in pred_triples]
    gold_direct = [normalize_triple(t, lang) for t in gold_triples]
    gold_swapped = [normalize_triple_swapped(t, lang) for t in gold_triples]

    remaining_gold_direct = Counter(gold_direct)
    remaining_gold_swapped = Counter(gold_swapped)
    tp = 0

    for pred in pred_norm:
        if remaining_gold_direct[pred] > 0:
            remaining_gold_direct[pred] -= 1
            swapped_key = (pred[2], pred[1], pred[0])
            if remaining_gold_swapped[swapped_key] > 0:
                remaining_gold_swapped[swapped_key] -= 1
            tp += 1
        elif remaining_gold_swapped[pred] > 0:
            remaining_gold_swapped[pred] -= 1
            direct_key = (pred[2], pred[1], pred[0])
            if remaining_gold_direct[direct_key] > 0:
                remaining_gold_direct[direct_key] -= 1
            tp += 1

    pred_total = len(pred_norm)
    gold_total = len(gold_direct)
    precision, recall, f1 = prf(tp, pred_total, gold_total)
    fp = pred_total - tp
    fn = gold_total - tp
    return {
        "triple_swap_aware_tp": tp,
        "triple_swap_aware_fp": fp,
        "triple_swap_aware_fn": fn,
        "triple_swap_aware_precision": precision,
        "triple_swap_aware_recall": recall,
        "triple_swap_aware_f1": f1,
    }


def component_overlap(pred_triples, gold_triples, key, lang=""):
    pred_vals = [normalize_token_text(t[key], lang) for t in pred_triples]
    gold_vals = [normalize_token_text(t[key], lang) for t in gold_triples]
    pred_counter = Counter(pred_vals)
    gold_counter = Counter(gold_vals)
    tp = sum((pred_counter & gold_counter).values())
    pred_total = sum(pred_counter.values())
    gold_total = sum(gold_counter.values())
    precision, recall, f1 = prf(tp, pred_total, gold_total)
    return {
        f"{key}_tp": tp,
        f"{key}_fp": pred_total - tp,
        f"{key}_fn": gold_total - tp,
        f"{key}_precision": precision,
        f"{key}_recall": recall,
        f"{key}_f1": f1,
    }


def jaccard_token_similarity(a, b, lang=""):
    ta = set(normalize_token_text(a, lang).split())
    tb = set(normalize_token_text(b, lang).split())
    if not ta and not tb:
        return 1.0
    if not ta or not tb:
        return 0.0
    return len(ta & tb) / len(ta | tb)


def soft_triple_similarity(pred_t, gold_t, lang=""):
    s_sim = jaccard_token_similarity(pred_t["subject"], gold_t["subject"], lang)
    p_sim = jaccard_token_similarity(pred_t["predicate"], gold_t["predicate"], lang)
    o_sim = jaccard_token_similarity(pred_t["object"], gold_t["object"], lang)
    return (s_sim + p_sim + o_sim) / 3.0


def soft_triple_similarity_swap_aware(pred_t, gold_t, lang=""):
    direct = soft_triple_similarity(pred_t, gold_t, lang)
    swapped_gold = {
        "subject": gold_t["object"],
        "predicate": gold_t["predicate"],
        "object": gold_t["subject"],
    }
    swapped = soft_triple_similarity(pred_t, swapped_gold, lang)
    return max(direct, swapped)


def greedy_soft_match(pred_triples, gold_triples, lang="", threshold=SOFT_MATCH_THRESHOLD, swap_aware=False):
    if not pred_triples and not gold_triples:
        prefix = "triple_soft_swap_aware" if swap_aware else "triple_soft"
        return {
            f"{prefix}_precision": 1.0,
            f"{prefix}_recall": 1.0,
            f"{prefix}_f1": 1.0,
            f"{prefix}_tp_weighted": 0.0,
            f"{prefix}_fp": 0,
            f"{prefix}_fn": 0,
            f"{prefix}_avg_match_score": 0.0,
        }

    remaining_gold = list(range(len(gold_triples)))
    sims = []

    for pred_t in pred_triples:
        best_idx = None
        best_sim = -1.0
        for gi in remaining_gold:
            if swap_aware:
                sim = soft_triple_similarity_swap_aware(pred_t, gold_triples[gi], lang)
            else:
                sim = soft_triple_similarity(pred_t, gold_triples[gi], lang)
            if sim > best_sim:
                best_sim = sim
                best_idx = gi
        if best_idx is not None and best_sim >= threshold:
            sims.append(best_sim)
            remaining_gold.remove(best_idx)
        else:
            sims.append(0.0)

    tp_weighted = float(sum(sims))
    pred_total = len(pred_triples)
    gold_total = len(gold_triples)
    precision = tp_weighted / pred_total if pred_total else 0.0
    recall = tp_weighted / gold_total if gold_total else 0.0
    f1 = (2 * precision * recall / (precision + recall)) if (precision + recall) else 0.0
    matched = sum(1 for x in sims if x >= threshold)
    fp = pred_total - matched
    fn = gold_total - matched
    prefix = "triple_soft_swap_aware" if swap_aware else "triple_soft"

    return {
        f"{prefix}_precision": float(precision),
        f"{prefix}_recall": float(recall),
        f"{prefix}_f1": float(f1),
        f"{prefix}_tp_weighted": tp_weighted,
        f"{prefix}_fp": fp,
        f"{prefix}_fn": fn,
        f"{prefix}_avg_match_score": float(np.mean([x for x in sims if x > 0]) if any(x > 0 for x in sims) else 0.0),
    }


def count_metrics(num_pred, num_gold, exact_tp, exact_fp, exact_fn):
    hallucinated = max(num_pred - exact_tp, 0)
    missed = max(num_gold - exact_tp, 0)
    ged_proxy = exact_fp + exact_fn
    norm_ged = safe_div(ged_proxy, max(num_pred + num_gold, 1))
    return {
        "hallucinated_triples": hallucinated,
        "missed_triples": missed,
        "pred_gold_ratio": safe_div(num_pred, num_gold, default=np.nan) if num_gold else np.nan,
        "count_abs_error": abs(num_pred - num_gold),
        "count_signed_error": num_pred - num_gold,
        "count_agreement": 1.0 - safe_div(abs(num_pred - num_gold), max(num_pred, num_gold, 1)),
        "hallucination_rate_pred": safe_div(hallucinated, num_pred),
        "omission_rate_gold": safe_div(missed, num_gold),
        "information_retention_rate": safe_div(exact_tp, num_gold),
        "information_injection_rate": safe_div(hallucinated, num_pred),
        "graph_edit_distance_proxy": ged_proxy,
        "normalized_graph_edit_distance": norm_ged,
        "graph_similarity": 1.0 - norm_ged,
    }


def evaluate_one_row(row):
    lang = row.get("lang", "")
    pred_triples = row["pred_triples"]
    gold_triples = row["gold_triples"]

    out = {}
    out.update(exact_triple_overlap(pred_triples, gold_triples, lang))
    out.update(component_overlap(pred_triples, gold_triples, "subject", lang))
    out.update(component_overlap(pred_triples, gold_triples, "predicate", lang))
    out.update(component_overlap(pred_triples, gold_triples, "object", lang))

    if USE_SWAP_AWARE_MATCH:
        out.update(exact_triple_overlap_swap_aware(pred_triples, gold_triples, lang))
        out["swap_gain_exact_f1"] = out["triple_swap_aware_f1"] - out["triple_exact_f1"]
        out["swap_gain_exact_recall"] = out["triple_swap_aware_recall"] - out["triple_exact_recall"]

    if USE_SOFT_MATCH:
        out.update(greedy_soft_match(pred_triples, gold_triples, lang, SOFT_MATCH_THRESHOLD, swap_aware=False))
        if USE_SWAP_AWARE_MATCH:
            out.update(greedy_soft_match(pred_triples, gold_triples, lang, SOFT_MATCH_THRESHOLD, swap_aware=True))
            out["swap_gain_soft_f1"] = out["triple_soft_swap_aware_f1"] - out["triple_soft_f1"]
            out["swap_gain_soft_recall"] = out["triple_soft_swap_aware_recall"] - out["triple_soft_recall"]

    out.update(
        count_metrics(
            num_pred=len(pred_triples),
            num_gold=len(gold_triples),
            exact_tp=out["triple_exact_tp"],
            exact_fp=out["triple_exact_fp"],
            exact_fn=out["triple_exact_fn"],
        )
    )
    return out


def micro_prf_for_group(group_df):
    tp = group_df["triple_exact_tp"].sum()
    fp = group_df["triple_exact_fp"].sum()
    fn = group_df["triple_exact_fn"].sum()
    pred_total = group_df["pred_total"].sum()
    gold_total = group_df["gold_total"].sum()

    p = tp / (tp + fp) if (tp + fp) else 0.0
    r = tp / (tp + fn) if (tp + fn) else 0.0
    f1 = (2 * p * r / (p + r)) if (p + r) else 0.0

    out = {
        "triple_exact_micro_precision": p,
        "triple_exact_micro_recall": r,
        "triple_exact_micro_f1": f1,
        "tp": tp,
        "fp": fp,
        "fn": fn,
        "hallucination_rate_micro": fp / pred_total if pred_total else 0.0,
        "omission_rate_micro": fn / gold_total if gold_total else 0.0,
        "graph_similarity_micro": 1.0 - ((fp + fn) / max(pred_total + gold_total, 1)),
    }

    if USE_SWAP_AWARE_MATCH and "triple_swap_aware_tp" in group_df.columns:
        swap_tp = group_df["triple_swap_aware_tp"].sum()
        swap_fp = group_df["triple_swap_aware_fp"].sum()
        swap_fn = group_df["triple_swap_aware_fn"].sum()
        swap_p = swap_tp / (swap_tp + swap_fp) if (swap_tp + swap_fp) else 0.0
        swap_r = swap_tp / (swap_tp + swap_fn) if (swap_tp + swap_fn) else 0.0
        swap_f1 = (2 * swap_p * swap_r / (swap_p + swap_r)) if (swap_p + swap_r) else 0.0
        out.update({
            "triple_swap_aware_micro_precision": swap_p,
            "triple_swap_aware_micro_recall": swap_r,
            "triple_swap_aware_micro_f1": swap_f1,
        })

    if USE_SOFT_MATCH and "triple_soft_tp_weighted" in group_df.columns:
        soft_tp = group_df["triple_soft_tp_weighted"].sum()
        soft_p = soft_tp / pred_total if pred_total else 0.0
        soft_r = soft_tp / gold_total if gold_total else 0.0
        soft_f1 = (2 * soft_p * soft_r / (soft_p + soft_r)) if (soft_p + soft_r) else 0.0
        out.update({
            "triple_soft_micro_precision": soft_p,
            "triple_soft_micro_recall": soft_r,
            "triple_soft_micro_f1": soft_f1,
        })

    if USE_SOFT_MATCH and USE_SWAP_AWARE_MATCH and "triple_soft_swap_aware_tp_weighted" in group_df.columns:
        soft_swap_tp = group_df["triple_soft_swap_aware_tp_weighted"].sum()
        soft_swap_p = soft_swap_tp / pred_total if pred_total else 0.0
        soft_swap_r = soft_swap_tp / gold_total if gold_total else 0.0
        soft_swap_f1 = (2 * soft_swap_p * soft_swap_r / (soft_swap_p + soft_swap_r)) if (soft_swap_p + soft_swap_r) else 0.0
        out.update({
            "triple_soft_swap_aware_micro_precision": soft_swap_p,
            "triple_soft_swap_aware_micro_recall": soft_swap_r,
            "triple_soft_swap_aware_micro_f1": soft_swap_f1,
        })

    return pd.Series(out)


# -----------------------
# Run evaluation
# -----------------------
df = load_generation_csvs()
print("Loaded:", df.shape)

records = []
for _, row in tqdm(df.iterrows(), total=len(df)):
    out = evaluate_one_row(row)
    base = row.to_dict()
    base.update(out)
    records.append(base)

eval_df = pd.DataFrame(records)
print("Evaluated:", eval_df.shape)


# -----------------------
# Save instance-level results
# -----------------------
instance_cols_first = [
    "model_name", "lang", "split", "category", "eid", "align_key",
    "num_gold_triples", "num_pred_triples",
    "gold_triples_text", "pred_triples_text",
    "triple_exact_precision", "triple_exact_recall", "triple_exact_f1",
    "triple_swap_aware_precision", "triple_swap_aware_recall", "triple_swap_aware_f1",
    "swap_gain_exact_f1", "swap_gain_exact_recall",
    "subject_precision", "subject_recall", "subject_f1",
    "predicate_precision", "predicate_recall", "predicate_f1",
    "object_precision", "object_recall", "object_f1",
    "hallucinated_triples", "missed_triples", "pred_gold_ratio",
    "count_abs_error", "count_signed_error", "count_agreement",
    "hallucination_rate_pred", "omission_rate_gold",
    "information_retention_rate", "information_injection_rate",
    "graph_edit_distance_proxy", "normalized_graph_edit_distance", "graph_similarity",
    "triple_soft_precision", "triple_soft_recall", "triple_soft_f1",
    "triple_soft_tp_weighted", "triple_soft_avg_match_score",
    "triple_soft_swap_aware_precision", "triple_soft_swap_aware_recall", "triple_soft_swap_aware_f1",
    "triple_soft_swap_aware_tp_weighted", "triple_soft_swap_aware_avg_match_score",
    "swap_gain_soft_f1", "swap_gain_soft_recall",
]
instance_cols = [c for c in instance_cols_first if c in eval_df.columns] + [c for c in eval_df.columns if c not in instance_cols_first]

eval_df.to_csv(OUTPUT_DIR / "instance_level_ie_metrics.csv", index=False)
eval_df.to_excel(OUTPUT_DIR / "instance_level_ie_metrics.xlsx", index=False)


# -----------------------
# Summary by model × language
# -----------------------
agg_spec = {
    "n": ("align_key", "size"),
    "triple_exact_precision": ("triple_exact_precision", "mean"),
    "triple_exact_recall": ("triple_exact_recall", "mean"),
    "triple_exact_f1": ("triple_exact_f1", "mean"),
    "triple_swap_aware_precision": ("triple_swap_aware_precision", "mean"),
    "triple_swap_aware_recall": ("triple_swap_aware_recall", "mean"),
    "triple_swap_aware_f1": ("triple_swap_aware_f1", "mean"),
    "swap_gain_exact_f1": ("swap_gain_exact_f1", "mean"),
    "swap_gain_exact_recall": ("swap_gain_exact_recall", "mean"),
    "subject_f1": ("subject_f1", "mean"),
    "predicate_f1": ("predicate_f1", "mean"),
    "object_f1": ("object_f1", "mean"),
    "hallucinated_triples": ("hallucinated_triples", "mean"),
    "missed_triples": ("missed_triples", "mean"),
    "pred_gold_ratio": ("pred_gold_ratio", "mean"),
    "count_abs_error": ("count_abs_error", "mean"),
    "count_signed_error": ("count_signed_error", "mean"),
    "count_agreement": ("count_agreement", "mean"),
    "hallucination_rate_pred": ("hallucination_rate_pred", "mean"),
    "omission_rate_gold": ("omission_rate_gold", "mean"),
    "information_retention_rate": ("information_retention_rate", "mean"),
    "information_injection_rate": ("information_injection_rate", "mean"),
    "graph_edit_distance_proxy": ("graph_edit_distance_proxy", "mean"),
    "normalized_graph_edit_distance": ("normalized_graph_edit_distance", "mean"),
    "graph_similarity": ("graph_similarity", "mean"),
}
if USE_SOFT_MATCH:
    agg_spec.update({
        "triple_soft_precision": ("triple_soft_precision", "mean"),
        "triple_soft_recall": ("triple_soft_recall", "mean"),
        "triple_soft_f1": ("triple_soft_f1", "mean"),
        "triple_soft_tp_weighted": ("triple_soft_tp_weighted", "mean"),
        "triple_soft_avg_match_score": ("triple_soft_avg_match_score", "mean"),
        "triple_soft_swap_aware_precision": ("triple_soft_swap_aware_precision", "mean"),
        "triple_soft_swap_aware_recall": ("triple_soft_swap_aware_recall", "mean"),
        "triple_soft_swap_aware_f1": ("triple_soft_swap_aware_f1", "mean"),
        "triple_soft_swap_aware_tp_weighted": ("triple_soft_swap_aware_tp_weighted", "mean"),
        "triple_soft_swap_aware_avg_match_score": ("triple_soft_swap_aware_avg_match_score", "mean"),
        "swap_gain_soft_f1": ("swap_gain_soft_f1", "mean"),
        "swap_gain_soft_recall": ("swap_gain_soft_recall", "mean"),
    })

summary_ml = (
    eval_df
    .groupby(["model_name", "lang"], dropna=False)
    .agg(**agg_spec)
    .reset_index()
    .sort_values(["model_name", "lang"])
)

for metric in [
    "triple_exact_f1", "triple_swap_aware_f1", "triple_soft_f1", "triple_soft_swap_aware_f1",
    "predicate_f1", "graph_similarity", "information_retention_rate"
]:
    if metric in summary_ml.columns:
        en_map = (
            summary_ml.loc[summary_ml["lang"] == "en", ["model_name", metric]]
            .drop_duplicates("model_name")
            .rename(columns={metric: f"{metric}_en"})
        )
        summary_ml = summary_ml.merge(en_map, on="model_name", how="left")
        summary_ml[f"{metric}_relative_to_en"] = summary_ml[metric] / summary_ml[f"{metric}_en"]

summary_ml.to_csv(OUTPUT_DIR / "summary_ie_by_model_lang.csv", index=False)
summary_ml.to_excel(OUTPUT_DIR / "summary_ie_by_model_lang.xlsx", index=False)

summary_mls = (
    eval_df
    .groupby(["model_name", "lang", "split"], dropna=False)
    .agg(**agg_spec)
    .reset_index()
    .sort_values(["model_name", "lang", "split"])
)
summary_mls.to_csv(OUTPUT_DIR / "summary_ie_by_model_lang_split.csv", index=False)
summary_mls.to_excel(OUTPUT_DIR / "summary_ie_by_model_lang_split.xlsx", index=False)

summary_mlc = (
    eval_df
    .groupby(["model_name", "lang", "category"], dropna=False)
    .agg(**agg_spec)
    .reset_index()
    .sort_values(["model_name", "lang", "category"])
)
summary_mlc.to_csv(OUTPUT_DIR / "summary_ie_by_model_lang_category.csv", index=False)
summary_mlc.to_excel(OUTPUT_DIR / "summary_ie_by_model_lang_category.xlsx", index=False)

micro_ml = (
    eval_df
    .groupby(["model_name", "lang"], dropna=False)
    .apply(micro_prf_for_group)
    .reset_index()
    .sort_values(["model_name", "lang"])
)
micro_ml.to_csv(OUTPUT_DIR / "summary_ie_micro_by_model_lang.csv", index=False)
micro_ml.to_excel(OUTPUT_DIR / "summary_ie_micro_by_model_lang.xlsx", index=False)

print("Saved outputs to:", OUTPUT_DIR.resolve())
print("Main files:")
print(" -", OUTPUT_DIR / "instance_level_ie_metrics.csv")
print(" -", OUTPUT_DIR / "summary_ie_by_model_lang.csv")
print(" -", OUTPUT_DIR / "summary_ie_micro_by_model_lang.csv")


Loaded: (44475, 41)


  0%|          | 0/44475 [00:00<?, ?it/s]

Evaluated: (44475, 104)
Saved outputs to: /home/vramon/notebooks/Rdf_text_EKAW/IE/evaluation/evaluation_outputs_ie
Main files:
 - evaluation_outputs_ie/instance_level_ie_metrics.csv
 - evaluation_outputs_ie/summary_ie_by_model_lang.csv
 - evaluation_outputs_ie/summary_ie_micro_by_model_lang.csv


/tmp/ipykernel_467860/141358403.py:640: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(micro_prf_for_group)
